There is evidence that stronger fractal patterns and nonlinearity are related to better health (Goldberger et al., 2002). To investigate this relationship we will calculate the HRV fractal patterns using the Detrended Fluctuation Analysis (DFA). The relevant health metrics here are sleep quality and the mental health questionnaires. The correlation results between the DFA and mental health scores should be very similar as to the results found for hypothesis 2. The correlation results between the DFA and the sleep quality scores should be very similar as to the results found for hypothesis 4.


Hypothesis: we expect individuals with better sleep quality to exhibit stronger fractal patterns in their HRV metrics, while individuals who score higher on the depression, anxiety and insomnia questionnaires will exhibit lower fractal patterns.

In [1]:
import pandas as pd
import numpy as np
import fathon
from fathon import fathonUtils as fu
from tqdm import tqdm
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

survey = pd.read_csv("data/survey.csv")
sensor_hrv = pd.read_csv("data/sensor_hrv.csv")
sensor_hrv_filtered = pd.read_csv("data/sensor_hrv_filtered.csv")
sleep_diary = pd.read_csv("data/sleep_diary.csv")

In [9]:
df_shf = sensor_hrv_filtered.copy()
df_shf["datetime"] = pd.to_datetime(df_shf["ts_start"], unit="ms", errors="coerce")

In [ ]:
# Compute DFA per deviceId

df_shf = sensor_hrv_filtered.copy()
df_shf["datetime"] = pd.to_datetime(df_shf["ts_start"], unit="ms", errors="coerce")


results = []

for pid, df_temp in tqdm(df_shf.groupby("deviceId", sort=True)):
    df_temp = df_temp.copy()

    # Clean ibi scores and get rid of NaNs
    x = df_temp["ibi"][np.isfinite(df_temp["ibi"])]

    # Calculate DFA using the fathon module
    profile = fu.toAggregated(x)
    dfa = fathon.DFA(profile)

    # Pick max window size of 2000 or the length of the profile // 4, whichever is smaller. Definitely can use some more literature
    win_max = min(2000, len(profile) // 4)

    wins = fu.linRangeByStep(10, win_max)
    n, F = dfa.computeFlucVec(wins, revSeg=True, polOrd=1)
    H, intercept = dfa.fitFlucVec()

    results.append(
        {
            "deviceId": pid,
            "H": H,
            "intercept": intercept,
            "n_points": len(x),
        }
    )

df_dfa = pd.DataFrame(results)
df_dfa.head()

100%|██████████| 49/49 [00:02<00:00, 18.38it/s]


,deviceId,H,intercept,n_points
0,ab60,0.626578,3.188126,1103
1,am77,0.879944,2.425040,285
2,av54,0.720930,2.824845,415
3,ba30,0.667620,3.320067,1045
4,bp16,0.874705,2.248134,528


In [3]:
# Merge DFA with sleep diary on participant level

df_sd = sleep_diary.copy()
df_sd.rename(columns={"userId":"deviceId"}, inplace=True)
df_sd = df_sd[["deviceId", "wakeup@night", "waso", "sleep_duration", "in_bed_duration", "sleep_latency", "sleep_efficiency"]]
df_sd_participant = df_sd.groupby("deviceId").mean()

df_analysis = df_dfa.merge(df_sd_participant, on="deviceId", how="inner")
df_analysis.head()

,deviceId,H,intercept,n_points,wakeup@night,waso,sleep_duration,in_bed_duration,sleep_latency,sleep_efficiency
0,ab60,0.626578,3.188126,1103,0.071429,3.214286,9.202381,9.708333,0.505952,0.947411
1,am77,0.879944,2.425040,285,0.142857,0.428571,7.801190,7.865476,0.064286,0.992028
2,av54,0.720930,2.824845,415,0.892857,35.892857,7.774405,8.398810,0.624405,0.925246
3,ba30,0.667620,3.320067,1045,0.321429,8.392857,7.842262,8.392857,0.550595,0.937293
4,bp16,0.874705,2.248134,528,0.357143,18.928571,8.294643,9.169048,0.874405,0.902251


In [4]:
# Merge DFA+Sleep Diary summary with survey questionnaires

df_s = survey.copy()
df_s = df_s[["deviceId", "ISI_1", "ISI_2", "ISI_F", "PHQ9_1", "PHQ9_2", "PHQ9_F", "GAD7_1", "GAD7_2", "GAD7_F"]]
df_s.head()

df_analysis = df_analysis.merge(df_s, on="deviceId", how="inner")
df_analysis.head()

,deviceId,H,intercept,n_points,wakeup@night,waso,sleep_duration,in_bed_duration,sleep_latency,sleep_efficiency,ISI_1,ISI_2,ISI_F,PHQ9_1,PHQ9_2,PHQ9_F,GAD7_1,GAD7_2,GAD7_F
0,ab60,0.626578,3.188126,1103,0.071429,3.214286,9.202381,9.708333,0.505952,0.947411,7,3.0,2.0,1,0.0,0.0,0,0.0,0.0
1,am77,0.879944,2.425040,285,0.142857,0.428571,7.801190,7.865476,0.064286,0.992028,1,5.0,4.0,4,5.0,6.0,2,3.0,5.0
2,av54,0.720930,2.824845,415,0.892857,35.892857,7.774405,8.398810,0.624405,0.925246,12,10.0,10.0,2,1.0,0.0,3,3.0,1.0
3,ba30,0.667620,3.320067,1045,0.321429,8.392857,7.842262,8.392857,0.550595,0.937293,8,8.0,3.0,1,7.0,1.0,2,2.0,0.0
4,bp16,0.874705,2.248134,528,0.357143,18.928571,8.294643,9.169048,0.874405,0.902251,6,3.0,4.0,3,2.0,0.0,4,1.0,1.0


In [6]:
# Correlate the DFA H score with various variables

def spearman_table(df, correlate, variables):
    results = []
    for v in variables:
        temp_df = df[[correlate, v]].dropna()
        rho, p = spearmanr(temp_df[correlate], temp_df[v])
        results.append({"variable": v, "rho": rho, "p": p})

    return pd.DataFrame(results)


correlation_variables = ["wakeup@night", "waso", "sleep_duration", "in_bed_duration", "sleep_latency", "sleep_efficiency", "ISI_1", "ISI_2", "ISI_F", "PHQ9_1", "PHQ9_2", "PHQ9_F", "GAD7_1", "GAD7_2", "GAD7_F"]

df_correlation = spearman_table(df_analysis, "H", correlation_variables)
df_correlation

,variable,rho,p
0,wakeup@night,0.106507,0.466380
1,waso,0.020213,0.890358
2,sleep_duration,-0.102551,0.483192
3,in_bed_duration,-0.192551,0.185004
4,sleep_latency,-0.095824,0.512494
5,sleep_efficiency,0.089388,0.541341
6,ISI_1,-0.076475,0.601483
7,ISI_2,0.055496,0.704883
8,ISI_F,0.110019,0.451727
9,PHQ9_1,0.067833,0.643286
